<a href="https://colab.research.google.com/github/Fares-pr0g/ML-journey-ep-2-Experimenting-with-NN-s-in-PyTorch/blob/main/Learning_PyTorch_07_RNN's.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project one – predicting the sentiment of IMDb movie reviews

In [1]:
# imports
import torch
from torch import nn


In [2]:
!pip install -q datasets

from datasets import load_dataset

imdb = load_dataset("stanfordnlp/imdb")


In [3]:
imdb

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [4]:
train_dataset = imdb["train"]
test_dataset = imdb["test"]

## Data Integrity Checks
Let's check for potential data issues like duplicates or unbalanced labels that could lead to the observed perfect accuracy.

In [5]:
# 1. Check for duplicates between training and validation/test sets
def check_duplicates(dataset1, dataset2, name1, name2):
    texts1 = [item['text'] for item in dataset1]
    texts2 = [item['text'] for item in dataset2]

    set1 = set(texts1)
    set2 = set(texts2)

    common_elements = set1.intersection(set2)
    if common_elements:
        print(f"WARNING: {len(common_elements)} duplicate text entries found between {name1} and {name2}.")
        # Optionally print some examples of duplicates
        # for i, text in enumerate(list(common_elements)[:5]):
        # print(f"  Example {i+1}: {text[:100]}...")
    else:
        print(f"No duplicate text entries found between {name1} and {name2}.")

print("--- Checking for duplicates between datasets ---")

check_duplicates(train_dataset, test_dataset, "Train Set", "Test Set")



--- Checking for duplicates between datasets ---


In [6]:
# check if there is same data points in the same dataset
def check_redundance(dataset, name):
    texts= [(item['text'], item['label']) for item in dataset]
    unique_texts= set(texts)
    if len(unique_texts) == len(texts):
        print(f"No redundant data points found in {name}.")
        return
    else:
      i=0
      for data_point in texts:
        if texts.count(data_point) >1:
          i+=1
          #print(f"WARNING: Redundant data point found in {name}: {data_point[0][:100]}...")
          #print(texts.count(data_point))
      return i

print("--- Checking for redundant data points ---")
check_redundance(train_dataset, "Train")
check_redundance(test_dataset, "Test")


--- Checking for redundant data points ---


390

In [7]:
# Let's remove the redundance in the train_dataset
def remove_redundant_entries(dataset):

    unique_data = set()
    cleaned_dataset_list = []

    for item in dataset:
        text = item['text']
        label = item['label']
        if (text, label) not in unique_data:
            unique_data.add((text, label))
            cleaned_dataset_list.append({'text': text, 'label': label})

    return cleaned_dataset_list

print("--- Removing redundant data points from the training dataset ---")
original_train_len = len(train_dataset)
train_dataset_cleaned_list = remove_redundant_entries(train_dataset)


from datasets import Dataset
train_dataset = Dataset.from_list(train_dataset_cleaned_list)

print(f"Original training dataset size: {original_train_len}")
print(f"Cleaned training dataset size: {len(train_dataset)}")
print(f"Removed {original_train_len - len(train_dataset)} redundant entries.")



--- Removing redundant data points from the training dataset ---
Original training dataset size: 25000
Cleaned training dataset size: 24904
Removed 96 redundant entries.


In [8]:
# Let's remove the redundance and data leakage in the test_dataset

def remove_redundance_and_leakage(test_dataset, train_dataset):
  test_unique_data= set()
  train_unique_data= set([(item["text"],item["label"]) for item in train_dataset])
  clean_data=[]

  for item in test_dataset:
    text= item["text"]
    label= item["label"]
    if (text, label) not in test_unique_data and (text, label) not in train_unique_data:
      test_unique_data.add((text, label))
      clean_data.append({"text": text, "label": label})

  return clean_data

print("--- Removing leakage and redundant data points from the testing dataset ---")
original_test_len = len(test_dataset)
test_dataset_cleaned_list = remove_redundance_and_leakage(test_dataset, train_dataset)


from datasets import Dataset
test_dataset = Dataset.from_list(test_dataset_cleaned_list)

print(f"Original testing dataset size: {original_test_len}")
print(f"Cleaned testing dataset size: {len(test_dataset)}")
print(f"Removed {original_test_len - len(test_dataset)} redundant entries.")


--- Removing leakage and redundant data points from the testing dataset ---
Original testing dataset size: 25000
Cleaned testing dataset size: 24678
Removed 322 redundant entries.


In [9]:
# Let's check if the issue is fixed
check_duplicates(train_dataset, test_dataset, "Train Set", "Test Set")
check_redundance(train_dataset, "Train")
check_redundance(test_dataset, "Test")

No duplicate text entries found between Train Set and Test Set.
No redundant data points found in Train.
No redundant data points found in Test.


**Super!! Now let's advance to more interesting stuff**

### **Device Agnostic Code:**

In [10]:
# Device agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

## **First step: Preprocessing**

In [11]:
# Step 1:Create the datasets
import torch
from torch.utils.data.dataset import random_split

torch.manual_seed(42)
# the sum of the lengths must be exactly equal to the mother dataset
train_dataset, valid_dataset = random_split(train_dataset, [20000, len(train_dataset)-20000])


In [12]:
# Let's visulaize an example of data point:
train_dataset[0]

{'text': 'The only part lacking in this movie is Shue\'s part as the daughter wanting to follow in her "aunt\'s" footsteps as a daytime soap star. Otherwise it would be a perfect 10.<br /><br />It seems that every actor enjoyed their parts and overacting to fulfill their own enjoyment as well as the script - I have to wonder if a little ad lib\'ing wasn\'t taking place in parts. It was well cast and there are some classic lines that will stick with you.<br /><br />It\'s a fantastic movie everyone should see at least once. I\'d recommend not drinking anything that would sting coming out your nose.<br /><br />You\'ll definitely want to watch the last scene closely, \'Nurse Nan\' has a little secret she\'d rather not have shared with you.<br /><br />If you love daytime soaps or despise them, this move pokes fun in all the right places.',
 'label': 1}

In [13]:
# Step 2: find unique tokens (words)
import re
from collections import Counter

def tokenizer(text):
    text = re.sub(r'<[^>]*>', '', text)

    emoticons = re.findall(
        r'(?::|;|=)(?:-)?(?:\)|\(|D|P)',
        text.lower()
    )

    text = re.sub(r'[\W]+', ' ', text.lower()) + \
           ' '.join(emoticons).replace('-', '')

    return text.split()


token_counts = Counter()

for item in train_dataset:
    tokens = tokenizer(item["text"])
    token_counts.update(tokens)

print("Vocab-size:", len(token_counts))

Vocab-size: 69421


In [14]:
!pip install -q torchtext

In [15]:
# Step 3: encoding each unique token into integers
token_to_idx={"<pad>": 0, "<unk>": 1}
token_to_idx.update({token[0]: idx for idx, token in enumerate(token_counts.most_common(), start=2)})

In [16]:
# example test
print([token_to_idx[token] for token in ['this', 'is', 'an', 'example']])

[11, 7, 35, 472]


In [17]:
# Step 3-A: define the functions for transformation
text_pipeline = lambda txt: [token_to_idx.get(token, token_to_idx["<unk>"]) for token in tokenizer(txt)]

#This was the problem of my model outputing 100% accuracy (because the labels are already 1's and 0's)
#label_pipeline= lambda lbl: 1. if lbl=='pos' else 0.

In [18]:
# Step 3-B: wrap the encode and transformation function
import torch
from torch import nn
def collate_batch(batch):
  label_list, text_list, lengths = [], [], []
  for item in batch:
    text = item["text"]
    label = item["label"]
    label_list.append(float(label))
    processed_text= torch.tensor(text_pipeline(text), dtype=torch.int64)
    text_list.append(processed_text)
    lengths.append(processed_text.size(0))

  label_list= torch.tensor(label_list)
  lengths = torch.tensor(lengths)
  padded_text_list = nn.utils.rnn.pad_sequence(text_list, batch_first=True)

  return padded_text_list, label_list, lengths

# Take a small batch
from torch.utils.data import DataLoader
dataloader= DataLoader(train_dataset, batch_size=4,
                       shuffle=False, collate_fn= collate_batch)
next(iter(dataloader))

(tensor([[    2,    64,   172,  1905,     9,    11,    17,     7, 10389,    13,
            172,    15,     2,   545,  1826,     6,   765,     9,    41,  2918,
             13,  9123,    15,     4,  7070,  1994,   321,   923,     8,    60,
             29,     4,   402,   167,     8,   184,    12,   174,   277,   506,
             67,   529,     3,  4595,     6,  9319,    67,   205,  3128,    15,
             72,    15,     2,   229,    10,    28,     6,   586,    46,     4,
            118,  3203, 21848,  8180,   288,    21,   648,   270,     9,   529,
              8,    14,    72,   179,     3,    40,    26,    49,   342,   408,
             12,    80,  1219,    18,    22,     8,    13,     4,   778,    17,
            295,   140,    66,    32,   220,   281,    10,   226,   382,    24,
           2835,   231,    12,    60, 10390,   583,    45,   131,  3018,    22,
            234,   409,   181,     6,   105,     2,   232,   133,  3226,  3670,
          14609,    47,     4,   118,   

In [19]:
# Let's create the actual dataloaders now

BATCH_SIZE= 32
train_dl= DataLoader(train_dataset, batch_size=BATCH_SIZE,
                     shuffle=True, collate_fn=collate_batch)

valid_dl= DataLoader(valid_dataset, batch_size=BATCH_SIZE,
                     shuffle=False, collate_fn=collate_batch)

test_dl= DataLoader(test_dataset, batch_size=BATCH_SIZE,
                     shuffle=False, collate_fn=collate_batch)


In [20]:
# 2. Examine label distribution
from collections import Counter

def get_label_distribution(dataset, name):
    labels = [item['label'] for item in dataset]
    distribution = Counter(labels)
    print(f"--- Label distribution in {name} ---")
    for label, count in distribution.items():
        print(f"  Label {label}: {count} ({count/len(labels):.2%})")
    if len(distribution) == 1:
        print(f"WARNING: Only one label ({list(distribution.keys())[0]}) present in {name}. This will lead to trivial perfect accuracy.")

get_label_distribution(train_dl.dataset, "Train Set")
get_label_distribution(valid_dl.dataset, "Validation Set")
get_label_distribution(test_dl.dataset, "Test Set")

--- Label distribution in Train Set ---
  Label 1: 10040 (50.20%)
  Label 0: 9960 (49.80%)
--- Label distribution in Validation Set ---
  Label 0: 2472 (50.41%)
  Label 1: 2432 (49.59%)
--- Label distribution in Test Set ---
  Label 0: 12266 (49.70%)
  Label 1: 12412 (50.30%)


## Model 0: LSTM architecture

In [21]:
class RNN(nn.Module):

  def __init__(self, vocab_size, embed_dim, rnn_hidden_size, fc_hidden_size):
    super().__init__()
    self.embedding= nn.Embedding(vocab_size, embed_dim,
                                 padding_idx=0)
    self.rnn= nn.LSTM(embed_dim, rnn_hidden_size, batch_first=True)
    self.fc1= nn.Linear(rnn_hidden_size, fc_hidden_size)
    self.relu= nn.ReLU()
    self.fc2= nn.Linear(fc_hidden_size, 1)
    self.sigmoid= nn.Sigmoid()

  def forward(self, text, lengths):
    out= self.embedding(text)
    out= nn.utils.rnn.pack_padded_sequence(out, lengths.cpu().numpy(),
                                           enforce_sorted=False, batch_first= True)
    out, (hidden, cell)= self.rnn(out)
    out= hidden[-1]
    out= self.fc1(out)
    out= self.relu(out)
    out= self.fc2(out)
    out= self.sigmoid(out)
    return out



In [22]:
vocab_size= len(token_to_idx)
embed_dim= 20
rnn_hidden_size= 64
fc_hidden_size= 64
torch.manual_seed(42)

model_0= RNN(vocab_size, embed_dim, rnn_hidden_size, fc_hidden_size).to(device)

In [23]:
# functionizing the training loop
def train(model, optimizer, loss_fn,dataloader):
  model= model.to(device)
  model.train()
  total_acc, total_loss= 0,0
  for text_batch, label_batch, lengths in dataloader:
    text_batch= text_batch.to(device)
    label_batch= label_batch.to(device)
    lengths= lengths.to(device)
    optimizer.zero_grad()
    pred= model(text_batch,lengths)[:,0]
    loss= loss_fn(pred, label_batch)
    loss.backward()
    optimizer.step()
    total_acc += ((pred>=0.5)==label_batch).sum().item()
    total_loss += loss.item()*label_batch.size(0)
  return total_acc/len(dataloader.dataset), total_loss/len(dataloader.dataset)

def evaluate(model, loss_fn, dataloader):
  model= model.to(device)
  model.eval()
  total_acc, total_loss= 0,0
  with torch.inference_mode():
    for text_batch, label_batch, lengths in dataloader:
      text_batch= text_batch.to(device)
      label_batch= label_batch.to(device)
      lengths= lengths.to(device)
      pred= model(text_batch, lengths)[:,0]
      loss= loss_fn(pred, label_batch)
      total_acc += ((pred>=0.5)==label_batch).float().sum().item()
      total_loss += loss.item()*label_batch.size(0)
  return total_acc/len(dataloader.dataset), total_loss/len(dataloader.dataset)



In [24]:
loss_fn= nn.BCELoss()
optimizer= torch.optim.Adam(model_0.parameters(), lr=0.001)

EPOCHS= 10
torch.manual_seed(42)

from timeit import default_timer as timer
start_time= timer()

for epoch in range(EPOCHS):
  acc_train, loss_train= train(model_0, optimizer, loss_fn, train_dl)
  acc_valid, loss_valid= evaluate(model_0, loss_fn, valid_dl)
  print(f'Epoch {epoch} accuracy: {acc_train:.4f} ------- val_accuracy: {acc_valid:.4f}')

end_time= timer()
print(f"Total training time: {end_time- start_time:.3f} seconds")

Epoch 0 accuracy: 0.5978 ------- val_accuracy: 0.6183
Epoch 1 accuracy: 0.7034 ------- val_accuracy: 0.6882
Epoch 2 accuracy: 0.7887 ------- val_accuracy: 0.7924
Epoch 3 accuracy: 0.7957 ------- val_accuracy: 0.5895
Epoch 4 accuracy: 0.8536 ------- val_accuracy: 0.8440
Epoch 5 accuracy: 0.9041 ------- val_accuracy: 0.8546
Epoch 6 accuracy: 0.9284 ------- val_accuracy: 0.8650
Epoch 7 accuracy: 0.9467 ------- val_accuracy: 0.8569
Epoch 8 accuracy: 0.9586 ------- val_accuracy: 0.8548
Epoch 9 accuracy: 0.9715 ------- val_accuracy: 0.8679
Total training time: 306.395 seconds


In [25]:
acc_test, _ = evaluate(model_0, loss_fn, test_dl)
print(f'Test accuracy: {acc_test:.4f}')

Test accuracy: 0.8563


# Model 1: Bidirectional LSTM Architecture

In [27]:
class Bi_RNN(nn.Module):

  def __init__(self, vocab_size, embed_dim, rnn_hidden_size, fc_hidden_size):
    super().__init__()
    self.embedding= nn.Embedding(vocab_size, embed_dim,
                                 padding_idx=0)
    self.rnn= nn.LSTM(embed_dim, rnn_hidden_size, batch_first=True, bidirectional= True)
    self.fc1= nn.Linear(rnn_hidden_size*2, fc_hidden_size)
    self.relu= nn.ReLU()
    self.fc2= nn.Linear(fc_hidden_size, 1)
    self.sigmoid= nn.Sigmoid()

  def forward(self, text, lengths):
    out= self.embedding(text)
    out= nn.utils.rnn.pack_padded_sequence(out, lengths.cpu().numpy(),
                                           enforce_sorted=False, batch_first= True)
    _, (hidden, cell)= self.rnn(out)
    out= torch.cat((hidden[-2],hidden[-1]), dim=1)
    out= self.fc1(out)
    out= self.relu(out)
    out= self.fc2(out)
    out= self.sigmoid(out)
    return out

In [28]:
vocab_size= len(token_to_idx)
embed_dim= 20
rnn_hidden_size= 64
fc_hidden_size= 64
torch.manual_seed(42)

model_1= Bi_RNN(vocab_size, embed_dim, rnn_hidden_size, fc_hidden_size).to(device)

In [29]:
loss_fn= nn.BCELoss()
optimizer= torch.optim.Adam(model_1.parameters(), lr=0.001)

EPOCHS= 10
torch.manual_seed(42)

from timeit import default_timer as timer
start_time= timer()

for epoch in range(EPOCHS):
  acc_train, loss_train= train(model_1, optimizer, loss_fn, train_dl)
  acc_valid, loss_valid= evaluate(model_1, loss_fn, valid_dl)
  print(f'Epoch {epoch} accuracy: {acc_train:.4f} ------- val_accuracy: {acc_valid:.4f}')

end_time= timer()
print(f"Total training time: {end_time- start_time:.3f} seconds")

Epoch 0 accuracy: 0.6462 ------- val_accuracy: 0.7276
Epoch 1 accuracy: 0.7686 ------- val_accuracy: 0.6850
Epoch 2 accuracy: 0.8111 ------- val_accuracy: 0.8089
Epoch 3 accuracy: 0.8674 ------- val_accuracy: 0.8281
Epoch 4 accuracy: 0.8956 ------- val_accuracy: 0.8436
Epoch 5 accuracy: 0.9207 ------- val_accuracy: 0.8518
Epoch 6 accuracy: 0.9298 ------- val_accuracy: 0.8499
Epoch 7 accuracy: 0.9472 ------- val_accuracy: 0.8562
Epoch 8 accuracy: 0.9597 ------- val_accuracy: 0.8524
Epoch 9 accuracy: 0.9691 ------- val_accuracy: 0.8583
Total training time: 467.821 seconds


In [30]:
acc_test, _ = evaluate(model_1, loss_fn, test_dl)
print(f'Test accuracy: {acc_test:.4f}')

Test accuracy: 0.8508


-> Same result. Who WHo (disappointed sound effect)